In [0]:
from pyspark.sql import functions as F

base = "/Volumes/learning/bronze/raw"

customers_raw = (spark.read
                 .option("header", True)
                 .csv(base + "/customers/")
                 .select(
                     "*",
                     F.current_timestamp().alias("_ingest_timestamp"),
                     F.col("_metadata.file_path").alias("_source_file"),
                 )
                )

customers_raw.printSchema()
print(customers_raw.count())
display(customers_raw.filter(F.col("customer_id").isin("6", "11", "21")))

In [0]:
(customers_raw.write
    .format("delta")                 # the default on Databricks anyway, but be explicit
    .mode("overwrite")               # safe to re-run while learning (explained below)
    .saveAsTable("learning.bronze.customers"))

In [0]:
%sql
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT _ingest_timestamp) AS load_batches,
       MIN(_source_file) AS source_file
FROM learning.bronze.customers;

In [0]:
%sql
DESCRIBE DETAIL learning.bronze.customers;

In [0]:
%sql
DESCRIBE HISTORY learning.bronze.customers;

In [0]:
from delta.tables import DeltaTable

customers = DeltaTable.forName(spark, "learning.bronze.customers")

customers.update(
    condition="customer_id = 11",
    set={"country": "'Germany'"}
)

customers.update(
    condition="customer_id = 12",
    set={"country": "null"}
)

In [0]:
display(customers.toDF())


In [0]:
%sql
DESCRIBE DETAIL learning.bronze.customers;
    
    


In [0]:
%sql
DESCRIBE HISTORY learning.bronze.customers;


In [0]:
%sql
SELECT customer_id, country FROM learning.bronze.customers VERSION AS OF 0
WHERE customer_id IN ("11", "12");

In [0]:
%sql
SELECT customer_id, country FROM learning.bronze.customers VERSION AS OF 6
WHERE customer_id IN ("11", "12");

In [0]:
%sql
RESTORE TABLE learning.bronze.customers TO VERSION AS OF 0

In [0]:
%sql
SELECT customer_id, country FROM learning.bronze.customers
WHERE customer_id IN ("11", "12");

In [0]:
%sql
SELECT customer_id, country FROM learning.bronze.customers VERSION AS OF 7
WHERE customer_id IN ("11", "12");

In [0]:
%sql
SELECT customer_id, country FROM learning.bronze.customers WHERE customer_id IN ("11", "12");
DESCRIBE HISTORY learning.bronze.customers;

In [0]:
from pyspark.sql import functions  as F

base = "/Volumes/learning/bronze/raw"

def ingest_to_bronze(entity: str) -> int:
    """Raw CSV folder -> learning.bronze.<entity>, all columns STRING + audit columns."""
    df = (
        spark
        .read
        .csv(f"{base}/{entity}", header=True)
        .select("*",
                F.current_timestamp().alias("_ingest_timestamp"),
                F.col("_metadata.file_path").alias("_source_file")))
    (df.write.format("delta").mode("overwrite")
        .saveAsTable(f"learning.bronze.{entity}"))
    n = spark.table(f"learning.bronze.{entity}").count()
    print(f"ingested {n} records to learning.bronze.{entity}")
    return n

for entity in ["customers", "orders"]:
    ingest_to_bronze(entity)            

In [0]:
%sql
SELECT order_id, customer_id, quantity, order_timestamp, status
FROM learning.bronze.orders
WHERE order_timestamp IN ("not_a_date", "2026-13-45 25:61:00")
OR customer_id NOT IN (SELECT customer_id FROM learning.bronze.customers)
OR quantity in ("-5", 0)
OR status <> lower(trim(status))
    
ORDER BY CAST(order_id AS INT)